In [1]:
import numpy as np
import pandas as pd
from numpy import kron, trace, sqrt
from functools import reduce
import dill
import sympy as sp
from repeater_helper import merge_state_GHZ, merge_state_W, proj_W_to_bell, get_final_state_G_G_G, get_final_state_G_G_W, normalize, get_final_state_G_W_W
q = sp.Symbol("q")
p_r = sp.Symbol("p_r")
p_l = sp.Symbol("p_l")

H = np.array([[1 / sqrt(2), 1 / sqrt(2)], [1 / sqrt(2), -1 / sqrt(2)]])
H3 = np.kron(np.kron(H, H), H)
def kron_all(*ops):
    return reduce(kron, ops)
def normalize(dm):
    return dm / trace(dm)
with open("atom_based_prob_click.dill", "rb") as f:
    prob_click = dill.load(f)

with open("atom_based_ion_dm.dill", "rb") as f:
    ion_dm = dill.load(f)

get_prob_click = sp.lambdify([q, p_r, p_l], prob_click)
get_ion_dm = sp.lambdify([q, p_r, p_l], ion_dm)
identity = np.array([[1, 0], [0, 1]])
x_gate = np.array([[0, 1], [1, 0]])
y_gate = np.array([[0, -1j], [1j, 0]])
z_gate = np.array([[1, 0], [0, -1]])
pauli_gates = [identity, x_gate, y_gate, z_gate]

proj_0 = np.array([[1, 0], [0, 0]])
proj_1 = np.array([[0, 0], [0, 1]])
ket_0 = np.array([[1], [0]])
ket_1 = np.array([[0], [1]])

U_03 = kron_all(proj_0, identity, identity, identity, identity, identity) + kron_all(
    proj_1, identity, identity, x_gate, identity, identity
)
U_14 = kron_all(identity, proj_0, identity, identity, identity, identity) + kron_all(
    identity, proj_1, identity, identity, x_gate, identity
)
U_25 = kron_all(identity, identity, proj_0, identity, identity, identity) + kron_all(
    identity, identity, proj_1, identity, identity, x_gate
)

ket_abc000 = kron_all(identity, identity, identity, ket_0, ket_0, ket_0)
ket_abc111 = kron_all(identity, identity, identity, ket_1, ket_1, ket_1)


def apply_AD(dm):
    input_dm = kron(dm, dm)
    output_dm = U_03 @ (U_14 @ (U_25 @ input_dm @ U_25.T) @ U_14.T) @ U_03.T
    output_dm = (ket_abc000.T @ output_dm @ ket_abc000) + (
        ket_abc111.T @ output_dm @ ket_abc111
    )
    prob_suc = trace(output_dm)
    return normalize(output_dm), prob_suc
def get_phase_QBER(dm):
    """
    Extract the phase error rate for the GHZ-state protocol given a density matrix.
    The density matrix should be in the DIAGONAL basis.
    """

    rotated_dm = H3 @ dm @ H3

    # XXX = 1
    # Components p_000, p_011, p_101, p_110
    XXX_p = rotated_dm[0, 0] + rotated_dm[3, 3] + rotated_dm[5, 5] + rotated_dm[6, 6]

    # XXX = -1
    # Components p_111, p_001, p_010, p_100
    XXX_m = 1 - XXX_p

    XXX_exp = XXX_p - XXX_m

    QBER = (1 -  XXX_exp) / 2

    return QBER


def get_bit_QBER(dm):
    """
    Extract the bit error rate of the GHZ protocol given a density matrix.
    The density matrix should be in the COMPUTATIONAL basis.
    """
    
    # Z_1 Z_2 = 1 (Alice and Bob are the same)
    # Components p_000, p_001, p_110, p_111

    ZZ_AB_p = dm[0, 0] + dm[1, 1] + dm[6, 6] + dm[7, 7]

    # Z_1 Z_2 = -1 (Alice and Bob are different)
    # Components p_010, p_011, p_100, p_101

    ZZ_AB_m = 1 - ZZ_AB_p

    # Z_1 Z_3 = 1 (Alice and Charlie are the same)
    # Components p_000, p_101, p_010, p_111

    ZZ_AC_p = dm[0, 0] + dm[2, 2] + dm[5, 5] + dm[7, 7]

    # Z_1 Z_3 = -1 (Alice and Charlie are different)
    ZZ_AC_m = 1 - ZZ_AC_p

    ZZ_AB_exp = ZZ_AB_p - ZZ_AB_m
    ZZ_AC_exp = ZZ_AC_p - ZZ_AC_m

    QBER_AB = float((1 - ZZ_AB_exp) / 2)
    QBER_AC = float((1 - ZZ_AC_exp) / 2)

    return max(QBER_AB, QBER_AC)


def get_bin_entropy(p):
    """
    Return the binary entropy.
    """
    if p == 0 or p == 1:
        return 0
    else:
        return (-p * (np.log2(p))) + (-(1 - p) * np.log2(1 - p))


def get_loss_ratio(distance_km, loss_db_per_km=0.3):
    """
    Calculate signal loss percentage over a given distance in km,
    based on attenuation in dB/km.

    Parameters:
    - distance_km: float or np.ndarray
    - loss_db_per_km: float, default is 0.3 dB/km

    Returns:
    - loss_percent: float or np.ndarray
    """
    total_loss_db = loss_db_per_km * distance_km
    transmission_ratio = 10 ** (-total_loss_db / 10)
    loss_ratio = 1 - transmission_ratio
    return loss_ratio


def get_sk_fraction(dm):
    bit_QBER = get_bit_QBER(dm)
    phase_QBER = get_phase_QBER(dm)

    return (
        max(1 - get_bin_entropy(phase_QBER) - get_bin_entropy(bit_QBER), 0)
    )


def get_key_rate_GGG(d, q_val, p_l_val, num_level = 2):
    t_ion = 100
    t_cnot = 100
    d_ES = d / (2 ** (num_level + 1))
    t_signal = 2 * d_ES / C
    t_meas = 100

    p_r_val = get_loss_ratio(d_ES)
    dm_val = get_ion_dm(q_val, p_r_val, p_l_val)
    prob_click = get_prob_click(q_val, p_r_val, p_l_val)

    t_end = (t_ion + t_signal) / prob_click
    t_merge_ops = t_cnot + t_meas 
    t_merge_signal = 2 * t_signal
    t_merge = 2 * (t_merge_ops + t_merge_signal)

    # Nesting level 1
    dm_W, prob_W = merge_state_W(dm_val)
    dm_GHZ, prob_GHZ = merge_state_GHZ(dm_val)
    prob_merge_level_1 = (prob_GHZ * prob_GHZ * prob_GHZ)
    t_end = ((11/6) * t_end + t_merge) / prob_merge_level_1
    t_merge_signal = t_merge_signal * 2 
    t_merge = 2 * (t_merge_ops + t_merge_signal)

    # Nesting level 2
    prob_merge_level_2 = 1
    dm_GHZ = normalize(get_final_state_G_G_G(dm_GHZ))
    t_end = (3 * t_end + t_merge) / prob_merge_level_2

    dm_GHZ, prob_AD = apply_AD(dm_GHZ)

    t_round = t_end + t_meas
    
    sk_rate = (1/2) * (prob_AD) *(get_sk_fraction(dm_GHZ) / t_round) * (10 ** 6)
    return sk_rate

def get_key_rate_GGW(d, q_val, p_l_val, num_level = 2):
    t_ion = 100
    t_cnot = 100
    d_ES = d / (2 ** (num_level + 1))
    t_signal = 2 * d_ES / C
    t_meas = 100

    p_r_val = get_loss_ratio(d_ES)
    dm_val = get_ion_dm(q_val, p_r_val, p_l_val)
    prob_click = get_prob_click(q_val, p_r_val, p_l_val)

    t_end = (t_ion + t_signal) / prob_click
    t_merge_ops = t_cnot + t_meas 
    t_merge_signal = 2 * t_signal
    t_merge = 2 * (t_merge_ops + t_merge_signal)

    # Nesting level 1
    dm_W, prob_W = merge_state_W(dm_val)
    dm_GHZ, prob_GHZ = merge_state_GHZ(dm_val)
    prob_merge_level_1 = (prob_GHZ * prob_GHZ * prob_W) * 3
    t_end = ((11/6) * t_end + t_merge) / prob_merge_level_1
    t_merge_signal = t_merge_signal * 2 
    t_merge = 2 * (t_merge_ops + t_merge_signal)

    # Nesting level 2
    dm_GHZ, prob_merge_level_2 = get_final_state_G_G_W(dm_GHZ, dm_W)
    t_end = (3 * t_end + t_merge) / prob_merge_level_2

    dm_GHZ, prob_AD = apply_AD(dm_GHZ)

    t_round = t_end + t_meas
    
    sk_rate = (1/2) * (prob_AD) *(get_sk_fraction(dm_GHZ) / t_round) * (10 ** 6)
    return sk_rate

def get_key_rate_GWW(d, q_val, p_l_val, num_level = 2):
    t_ion = 100
    t_cnot = 100
    d_ES = d / (2 ** (num_level + 1))
    t_signal = 2 * d_ES / C
    t_meas = 100

    p_r_val = get_loss_ratio(d_ES)
    dm_val = get_ion_dm(q_val, p_r_val, p_l_val)
    prob_click = get_prob_click(q_val, p_r_val, p_l_val)

    t_end = (t_ion + t_signal) / prob_click
    t_merge_ops = t_cnot + t_meas 
    t_merge_signal = 2 * t_signal
    t_merge = 2 * (t_merge_ops + t_merge_signal)

    # Nesting level 1
    dm_W, prob_W = merge_state_W(dm_val)
    dm_GHZ, prob_GHZ = merge_state_GHZ(dm_val)
    prob_merge_level_1 = (prob_GHZ * prob_W * prob_W) * 3
    t_end = ((11/6) * t_end + t_merge) / prob_merge_level_1
    t_merge_signal = t_merge_signal * 2 
    t_merge = 2 * (t_merge_ops + t_merge_signal)

    # Nesting level 2
    dm_GHZ, prob_merge_level_2 = get_final_state_G_W_W(dm_GHZ, dm_W)
    t_end = (3 * t_end + t_merge) / prob_merge_level_2

    dm_GHZ, prob_AD = apply_AD(dm_GHZ)

    t_round = t_end + t_meas
    
    sk_rate = (1/2) * (prob_AD) *(get_sk_fraction(dm_GHZ) / t_round) * (10 ** 6)
    return sk_rate

# Speed of light in km / us
C = 0.2
df_p_l_0_01 = pd.read_csv('../../W_data/atom_based_p_l_0_01.csv')
df_p_l_0_1 = pd.read_csv('../../W_data/atom_based_p_l_0_1.csv')
df_p_l_0_5 = pd.read_csv('../../W_data/atom_based_p_l_0_5.csv')

d_lst = df_p_l_0_01['distance_km'].to_numpy()
opt_qs_level_2_p_l_0_01 = df_p_l_0_01['opt_qs_level_2'].to_numpy()
opt_qs_level_2_p_l_0_1 = df_p_l_0_1['opt_qs_level_2'].to_numpy()
opt_qs_level_2_p_l_0_5 = df_p_l_0_5['opt_qs_level_2'].to_numpy()

p_l_val = 0.01


key_rates_level_2 = np.zeros_like(d_lst)
for i in range(len(d_lst)):
    sk = get_key_rate_GGG(d_lst[i], opt_qs_level_2_p_l_0_01[i], p_l_val= p_l_val, num_level= 2) + get_key_rate_GGW(d_lst[i], opt_qs_level_2_p_l_0_01[i], p_l_val= p_l_val, num_level= 2) + get_key_rate_GWW(d_lst[i], opt_qs_level_2_p_l_0_01[i], p_l_val= p_l_val, num_level= 2)
    key_rates_level_2[i] = sk

df = pd.DataFrame({'distance_km': d_lst, 
                   'keyrate_bps_level_2': key_rates_level_2})

df.to_csv('atom_based_GHZ_p_l_0_01_level_2_AD.csv', index = False, sep = ',')


p_l_val = 0.1


key_rates_level_2 = np.zeros_like(d_lst)
for i in range(len(d_lst)):
    sk = get_key_rate_GGG(d_lst[i], opt_qs_level_2_p_l_0_1[i], p_l_val= p_l_val, num_level= 2) + get_key_rate_GGW(d_lst[i], opt_qs_level_2_p_l_0_1[i], p_l_val= p_l_val, num_level= 2) + get_key_rate_GWW(d_lst[i], opt_qs_level_2_p_l_0_1[i], p_l_val= p_l_val, num_level= 2)
    key_rates_level_2[i] = sk

df = pd.DataFrame({'distance_km': d_lst, 
                   'keyrate_bps_level_2': key_rates_level_2})

df.to_csv('atom_based_GHZ_p_l_0_1_level_2_AD.csv', index = False, sep = ',')

p_l_val = 0.5


key_rates_level_2 = np.zeros_like(d_lst)
for i in range(len(d_lst)):
    sk = get_key_rate_GGG(d_lst[i], opt_qs_level_2_p_l_0_5[i], p_l_val= p_l_val, num_level= 2) + get_key_rate_GGW(d_lst[i], opt_qs_level_2_p_l_0_5[i], p_l_val= p_l_val, num_level= 2) + get_key_rate_GWW(d_lst[i], opt_qs_level_2_p_l_0_5[i], p_l_val= p_l_val, num_level= 2)
    key_rates_level_2[i] = sk

df = pd.DataFrame({'distance_km': d_lst, 
                   'keyrate_bps_level_2': key_rates_level_2})

df.to_csv('atom_based_GHZ_p_l_0_5_level_2_AD.csv', index = False, sep = ',')